# Emerging Market v7 — Data Prep with IBNR + A v E + New Business

Key changes from v6:
1. **Data-driven development**: Uses developed market count LDFs (emerging too sparse) + emerging paid/incurred severity LDFs
2. **A v E offset**: Computes 2023 loss ratios by line (first full year for emerging), creates expected_claim
3. **Year excluded from features** (handled in modeling notebook)
4. **New business flag preserved** for new business loading in modeling
5. All v6 cleaning retained: sum_insured, construction type harmonization (BRK->BRICK etc.)

In [0]:
# Step 1: Load raw data
import pandas as pd
import numpy as np
import os

BASE = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026'
DATA_DIR = f'{BASE}/data_emerging/'
V7_DIR = f'{BASE}/files_v7_emerging/'
os.makedirs(V7_DIR, exist_ok=True)

files_to_load = [
    'policies_motor', 'policies_buildings', 'policies_contents',
    'test_motor', 'test_buildings', 'test_contents',
    'claims', 'customers', 'addresses', 'weather', 'sample_submission'
]
dfs = {}
for name in files_to_load:
    dfs[name] = pd.read_parquet(os.path.join(DATA_DIR, f'{name}.parquet'))
    print(f'{name}: {dfs[name].shape}')
print('\nAll files loaded.')

In [0]:
# Step 2: Data cleaning (v6 fixes + emerging harmonization)

# 2a. Exposure + Year
for name in ['policies_motor', 'policies_buildings', 'policies_contents',
             'test_motor', 'test_buildings', 'test_contents']:
    df = dfs[name]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year

# 2b. Construction type harmonization (emerging codes)
CONSTRUCTION_MAP_EMG = {'BRK': 'BRICK', 'STN': 'STONE', 'WOD': 'TIMBER', 'MIX': 'MIXED'}
for name in ['policies_buildings', 'test_buildings']:
    df = dfs[name]
    df['construction_type'] = df['construction_type'].astype(str).str.strip().str.upper().replace(CONSTRUCTION_MAP_EMG)

# 2c. Strip ID columns
id_cols = ['policy_id', 'customer_id', 'address_id', 'claim_id', 'row_id']
for name, df in dfs.items():
    for col in id_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# 2d. Drop compulsory_excess if constant
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    if 'compulsory_excess' in dfs[name].columns and dfs[name]['compulsory_excess'].nunique() == 1:
        dfs[name].drop('compulsory_excess', axis=1, inplace=True)

# 2e. SUM_INSURED FIX
for name in ['policies_buildings', 'policies_contents', 'test_buildings', 'test_contents']:
    df = dfs[name]
    if 'sum_insured' not in df.columns: continue
    line = 'buildings' if 'buildings' in name else 'contents'
    threshold = 10_000_000 if line == 'buildings' else 1_000_000
    flagged = df['sum_insured'] >= threshold
    if flagged.sum() > 0:
        df.loc[flagged, 'sum_insured'] = df.loc[flagged, 'sum_insured'] / 1000
        print(f'{name}: {flagged.sum()} sum_insured values fixed')

# 2f. ADD AGE_BAND (if not already present)
def to_age_band(age):
    if pd.isna(age): return None
    if age <= 24: return '17-24'
    elif age <= 34: return '25-34'
    elif age <= 49: return '35-49'
    elif age <= 64: return '50-64'
    else: return '65+'
for name in ['policies_motor', 'test_motor']:
    if 'driver_age_band' not in dfs[name].columns and 'driver_age' in dfs[name].columns:
        dfs[name]['driver_age_band'] = dfs[name]['driver_age'].apply(to_age_band)
for name in ['policies_buildings', 'policies_contents', 'test_buildings', 'test_contents']:
    if 'policyholder_age_band' not in dfs[name].columns and 'policyholder_age' in dfs[name].columns:
        dfs[name]['policyholder_age_band'] = dfs[name]['policyholder_age'].apply(to_age_band)

print('Step 2 complete.')

In [0]:
# Step 3: Join enrichment data
customers_clean = dfs['customers'].drop(columns=['surname'], errors='ignore')
addresses_clean = dfs['addresses'].copy()
weather = dfs['weather'].copy()
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

policy_files = {'motor': 'policies_motor', 'buildings': 'policies_buildings', 'contents': 'policies_contents'}
for line, fname in policy_files.items():
    df = dfs[fname]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses_clean, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    dfs[fname] = df
    print(f'{line}: {df.shape}')

In [0]:
# Step 4: Claims processing
claims = dfs['claims']
claims['incurred_net_of_recoveries'] = claims['incurred_to_date'] - claims['recoveries']
claims['loss_year'] = pd.to_datetime(claims['loss_date']).dt.year
claims['report_date_dt'] = pd.to_datetime(claims['report_date'], errors='coerce')
claims['report_year'] = claims['report_date_dt'].dt.year
claims['dev_lag'] = claims['report_year'] - claims['loss_year']
claims['paid_incurred_ratio'] = claims['paid_to_date'] / claims['incurred_to_date'].clip(lower=0.01)
claims_active = claims[claims['status'] != 'repudiated'].copy()
print(f'Active claims: {len(claims_active)}')
for yr in sorted(claims_active['loss_year'].unique()):
    yr_c = claims_active[claims_active['loss_year'] == yr]
    print(f'  {yr}: {len(yr_c)} claims, {(yr_c["status"]=="open").mean():.1%} open, paid/incurred={yr_c["paid_incurred_ratio"].mean():.3f}')

In [0]:
# Step 5: Data-driven development
# Emerging claim count triangle is too sparse for reliable chain-ladder (2021: 6 claims, 2022: 28)
# Use DEVELOPED market count LDFs + EMERGING market severity LDFs
# NOTE: Buildings floor of 1.20 tested but HURT Kaggle score (0.14443 -> 0.13858).
# The data-driven LDFs without floor are more accurate. Do not apply a floor.

# Developed market count LDFs (from data prep notebook output)
COUNT_LDF_BY_DEVLAG = {0: 1.0545, 1: 1.0027, 2: 1.0001, 3: 1.0}
print(f'Using developed market count LDFs: {COUNT_LDF_BY_DEVLAG}')

# Emerging severity LDFs from paid/incurred ratio
pi_by_year = claims_active.groupby('loss_year')['paid_incurred_ratio'].mean()
severity_ldf_by_year = (1 / pi_by_year).clip(lower=1.0)
print(f'\nEmerging severity LDFs by year:')
for yr, ldf in severity_ldf_by_year.items():
    print(f'  {yr}: paid/incurred={pi_by_year[yr]:.4f}, LDF={ldf:.4f}')

# Combined LDF: count_LDF (by dev_age_at_snapshot) * severity_LDF (by loss_year)
SNAPSHOT_YEAR = 2024
claims_active['dev_age_at_snapshot'] = SNAPSHOT_YEAR - claims_active['loss_year']
claims_active['count_ldf'] = claims_active['dev_age_at_snapshot'].map(COUNT_LDF_BY_DEVLAG).fillna(1.0)
claims_active['severity_ldf'] = claims_active['loss_year'].map(severity_ldf_by_year).fillna(1.0)
claims_active['combined_ldf'] = claims_active['count_ldf'] * claims_active['severity_ldf']
claims_active['ultimate_claims'] = claims_active['incurred_net_of_recoveries'] * claims_active['combined_ldf']

print(f'\n=== Combined LDF by year ===')
for yr in sorted(claims_active['loss_year'].unique()):
    yr_c = claims_active[claims_active['loss_year'] == yr]
    print(f'  {yr}: count_LDF={yr_c["count_ldf"].mean():.4f}, sev_LDF={yr_c["severity_ldf"].mean():.4f}, combined={yr_c["combined_ldf"].mean():.4f}')

print(f'\nTotal ultimate: {claims_active["ultimate_claims"].sum():,.0f}')
print(f'Total incurred: {claims_active["incurred_net_of_recoveries"].sum():,.0f}')
print(f'Uplift: {(claims_active["ultimate_claims"].sum()/claims_active["incurred_net_of_recoveries"].sum()-1)*100:.1f}%')

In [0]:
# Step 5b: Match claims to policy rows and aggregate
claims_active['loss_date_dt'] = pd.to_datetime(claims_active['loss_date'])

for line, fname in policy_files.items():
    df = dfs[fname].copy()
    df['_ps_dt'] = pd.to_datetime(df['period_start'])
    df['_pe_dt'] = pd.to_datetime(df['period_end'])
    line_claims = claims_active[claims_active['line'] == line].copy()
    merged = line_claims.merge(df[['row_id', 'policy_id', '_ps_dt', '_pe_dt', 'Year']], on='policy_id', how='inner')
    merged = merged[merged['loss_date_dt'].dt.year == merged['Year']]
    mask = (merged['loss_date_dt'] >= merged['_ps_dt']) & (merged['loss_date_dt'] <= merged['_pe_dt'])
    matched = merged[mask].copy()
    matched = matched.sort_values('row_id').drop_duplicates('claim_id', keep='first')
    agg = matched.groupby('row_id').agg(
        ultimate_claims=('ultimate_claims', 'sum'),
        claim_count=('claim_id', 'count'),
        reported_incurred=('incurred_net_of_recoveries', 'sum'),
        open_claim_count=('status', lambda x: (x == 'open').sum()),
    ).reset_index()
    df = df.merge(agg, on='row_id', how='left')
    for c in ['ultimate_claims', 'reported_incurred']:
        df[c] = df[c].fillna(0)
    df['claim_count'] = df['claim_count'].fillna(0).astype(int)
    df['open_claim_count'] = df['open_claim_count'].fillna(0).astype(int)
    df.drop(['_ps_dt', '_pe_dt'], axis=1, inplace=True)
    dfs[fname] = df
    print(f'{line}: {len(df)} rows, {(df["claim_count"]>0).sum()} with claims, mean UC={df["ultimate_claims"].mean():.4f}')

In [0]:
# Step 6: A v E — compute 2023 loss ratios + create expected_claim + save proc files
# Use 2023 as the first full year for emerging (2021-2022 have only 6 and 28 claims)

LR_2023 = {}
for line in ['motor', 'buildings', 'contents']:
    df = pd.read_parquet(os.path.join(DATA_DIR, f'policies_{line}.parquet'))
    df['Year'] = pd.to_datetime(df['period_start']).dt.year
    df_2023 = df[df['Year'] == 2023].copy()
    ps = pd.to_datetime(df_2023['period_start'])
    pe = pd.to_datetime(df_2023['period_end'])
    df_2023['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    ep = (df_2023['exposure'] * df_2023['written_premium']).sum()
    
    c23 = claims_active[(claims_active['loss_year'] == 2023) & (claims_active['line'] == line)]
    ultimate_2023 = c23['ultimate_claims'].sum()
    lr = ultimate_2023 / ep if ep > 0 else 0.5
    LR_2023[line] = lr
    print(f'{line}: EP={ep:,.0f}, ultimate_2023={ultimate_2023:,.0f}, LR_2023={lr:.4f}, n_claims={len(c23)}')

print(f'\nLR_2023: {LR_2023}')

# Create expected_claim column in proc files
for line, fname in policy_files.items():
    df = dfs[fname]
    df['expected_claim'] = df['exposure'] * df['written_premium'] * LR_2023[line]
    df['ae_ratio'] = df['ultimate_claims'] / df['expected_claim'].clip(lower=0.01)
    print(f'{line}: expected_claim mean={df["expected_claim"].mean():.2f}, A/E mean={df["ae_ratio"].mean():.4f}')

# Save v7 proc files
for line, fname in policy_files.items():
    df = dfs[fname]
    out_path = os.path.join(V7_DIR, f'proc_{line}_v7.parquet')
    df.to_parquet(out_path, index=False)
    print(f'Saved: {out_path} ({len(df)} rows, {df.shape[1]} cols)')
    if 'sum_insured' in df.columns:
        thresh = 10_000_000 if line == 'buildings' else 1_000_000
        print(f'  sum_insured >= {thresh:,}: {(df["sum_insured"]>=thresh).sum()} (should be 0)')

import json
with open(os.path.join(V7_DIR, 'lr_2023.json'), 'w') as f:
    json.dump({k: float(v) for k, v in LR_2023.items()}, f)
print(f'\nSaved LR_2023 to {V7_DIR}lr_2023.json')

# Check new business ratio
print('\n=== New Business Ratio ===')
for line in ['motor', 'buildings', 'contents']:
    train_nb = dfs[f'policies_{line}']['new_business'].mean()
    test_nb = dfs[f'test_{line}']['new_business'].mean()
    print(f'  {line}: train {train_nb:.1%} new, test {test_nb:.1%} new')

print('\n=== v7 Emerging Data Prep Complete ===')